# Identification of Potential Post-Prenylation Processing Sites

For proteins containing Motif 2 with complete entries across variables X, Y, Z, and AA, we have predicted the presence of an internal prenylation site. A key follow-up question is whether these internal sites undergo standard post-prenylation processing, specifically the proteolytic cleavage of the amino acid chain immediately following the modified cysteine residue.

To investigate this, we are extracting the sequences of the potential new N-termini. These sequences are defined from the first amino acid following the cysteine residue up to the first downstream trypsin cleavage site.

These predicted neo-N-terminal sequences will be integrated into our database to facilitate further proteomic validation and downstream analysis.

# Setup

## Import and install Packages

In [ ]:
import sys
import os
import session_info

In [ ]:
import pandas as pd
import re

from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

In [ ]:
!pip install 'openpyxl'

In [ ]:
# Display session information
session_info.show()

## Set working directory

In [ ]:
cwd = os.getcwd()
if not cwd.endswith("Isoform_Database/Jupyter_environment/Isoform_Database_SIAF/04_Prenylation_Database"):
    print("WARNING: The working directory is not set to the '04_Prenylation_Database'!")
    print("Current working directory:", cwd)
else:
    print(f"Working directory is correctly set to the '04_Prenylation_Database' folder (\"{cwd}\").")

In [ ]:
# Data directories
unique_dir = "../01_UniProt/data/unique/"
duplicates_dir = "../01_UniProt/data/sequence_duplicates/"

## Read in Supplementary Table S1 and unique sequences

In [ ]:
df_prenylation = pd.read_excel('data/SupplementaryTableS1.xlsx')
full_proteome = pd.read_csv(os.path.join(unique_dir, 'full_proteome_unique.csv'))
duplicates_list = pd.read_csv(os.path.join(duplicates_dir, 'duplicates_list_01042026.csv'))

In [ ]:
#Take a quick look at supplementary table
df_prenylation

In [ ]:
# only keep proteins with internal prenylation motif (motif=2)
motif_2 = df_prenylation[df_prenylation['Motif'] == 2]
motif_2

In [ ]:
# check if any of these are in the duplicates list
motif_2['Protein_ID'].isin(duplicates_list['ID']).sum()

# Get Neo-N-termini
Function identifes the internal Cysteine and then scans the sequence downstream until it hits a Trypsin cleavage site.

In [ ]:
# Proteins that have a value for PositionC_Internal_pae0
internalC_pae0 = motif_2[motif_2["PositionC_Internal_pae0"].notna()]

# Proteins with position in lowest pae column
internalC_lowpae = motif_2[motif_2["PositionC_Internal_Lowest_pae"].notna()]

In [ ]:
# Merge the seq and internal C Position into new df
neo_n_pae0 = internalC_pae0.merge(
    full_proteome, 
    left_on='Protein_ID', 
    right_on='ID'
)

neo_n_lowpae = internalC_lowpae.merge(
    full_proteome, 
    left_on='Protein_ID', 
    right_on='ID'
)

# Clean up by selecting only the columns you need
# and ensuring the position is an integer
neo_n_pae0 = neo_n_pae0[['ID', 'PositionC_Internal_pae0', 'seq']]
neo_n_pae0['PositionC_Internal_pae0'] = neo_n_pae0['PositionC_Internal_pae0'].astype(int)

neo_n_lowpae = neo_n_lowpae[['ID', 'PositionC_Internal_Lowest_pae', 'seq']]
neo_n_lowpae['PositionC_Internal_Lowest_pae'] = neo_n_lowpae['PositionC_Internal_Lowest_pae'].astype(int)

In [ ]:
def get_fragment(row, pos):
    seq = row['seq']
    
    # Start right after the Cysteine
    start = pos + 1
    after_cys = seq[start:]

    if pos >= len(seq) or seq[pos].upper() != 'C':
        # You can return None, an empty string, or a custom error message
        return print("No C in given pos in canonical form")
        
    # Find the first Trypsin site (K or R not followed by P)
    match = re.search(r'[KR](?!P)', after_cys)
    
    if match:
        return after_cys[:match.end()]
    return after_cys # Return the rest if no trypsin site found

# Apply the function to create the new column
# Here, 'r' represents the row
neo_n_pae0['neo_n_terminus'] = neo_n_pae0.apply(
    lambda r: get_fragment(r, int(r['PositionC_Internal_pae0'])), 
    axis=1
)

neo_n_lowpae['neo_n_terminus'] = neo_n_lowpae.apply(
    lambda r: get_fragment(r, int(r['PositionC_Internal_Lowest_pae'])), 
    axis=1
)

In [ ]:
# Filter by fragment length
neo_n_pae0['fragment_length'] = neo_n_pae0['neo_n_terminus'].str.len()

neo_n_pae0_filtered = neo_n_pae0[
    (neo_n_pae0['fragment_length'] >= 7) & 
    (neo_n_pae0['fragment_length'] <= 52)
]


neo_n_lowpae['fragment_length'] = neo_n_lowpae['neo_n_terminus'].str.len()

neo_n_lowpae_filtered = neo_n_lowpae[
    (neo_n_lowpae['fragment_length'] >= 7) & 
    (neo_n_lowpae['fragment_length'] <= 52)
]

In [ ]:
neo_n_pae0_filtered[neo_n_pae0_filtered["ID"]=="P09110"]

In [ ]:
neo_n_lowpae_filtered.head()

Q9UMX0 (pos = -28) and Q9UDW1 (pos = -9) have no position in PositionC_Internal_pae0 or PositionC_Internal_Lowest_pae, but a position in PositionC_Internal_C1. Both proteins do not contain any Cystein in the UniProt sequence of the first isoform, but Q9UMX0 has a C in the  4th splice variant and Q9UDW1 has a C in the 2nd splice variant. These will be added to the databse.

## Create Fasta
The Fasta File for the database contains the ID and the neo_n_terminus.

In [ ]:
# 1. Prepare combined records list
# We use the final_records list from your existing header style
neo_n_records = []

# 2. Process neo_n_pae0
for _, row in neo_n_pae0_filtered.iterrows():
    iso_id = row['ID']
    pep_seq = row['neo_n_terminus']
    
    if pd.notna(pep_seq):
        # Using the UniProt-style header you provided
        rec = SeqRecord(
            Seq(pep_seq), 
            id=f"sp|{iso_id}|INTERNAL_PAE0", 
            description=""
        )
        neo_n_records.append(rec)

# 3. Process neo_n_lowpae
for _, row in neo_n_lowpae_filtered.iterrows():
    iso_id = row['ID']
    pep_seq = row['neo_n_terminus']
    
    if pd.notna(pep_seq):
        rec = SeqRecord(
            Seq(pep_seq), 
            id=f"sp|{iso_id}|INTERNAL_LOWPAE", 
            description=""
        )
        neo_n_records.append(rec)

# 4. Add Q9UMX0 and Q9UDW1 to neo_n_records
neo_n_records.append(
    SeqRecord(
        Seq('QESSNDAGDDEEPGPSFEQPRKHPR'), 
        id="sp|Q9UMX0-4|INTERNAL_C1", 
        description=""
    ) 
)

neo_n_records.append(
    SeqRecord(
        Seq('AIPDLGPA'), 
        id="sp|Q9UDW1-2|INTERNAL_C1", 
        description=""
    ) 
)
# 5. Write to file
output_filename = "data/Post-Prenylation_Processing_Database.fasta"
with open(output_filename, "w") as output_handle:
    SeqIO.write(neo_n_records, output_handle, "fasta")

print(f"FASTA created with {len(neo_n_records)} entries.")